# Annotate an AIRR query against VDJdb

Run ordinary human TRB annotation through the installed CLI. Keep the complete examples directory with this notebook. The bundled AIRR file is a demonstration query file. The two rows show how the reference and the resulting evidence are recorded.

By default the CLI fetches the current official VDJdb reference. Set `VDJMATCH_EXAMPLE_VDJDB` to a previously downloaded official VDJdb ZIP to run offline and pin its identity. Record the release and SHA-256 from the manifest before comparing any two runs. For a custom TSV and for labels without MHC metadata, see the [custom-reference notebook](annotation.ipynb).

In [ ]:
# Identify the installed CLI and print the fresh kernel environment.
import os, sys, subprocess, tempfile, json
from pathlib import Path
from importlib.metadata import version
os.environ["POLARS_MAX_THREADS"] = "1"
print("Python:", sys.version, "CLI interpreter:", sys.executable)
for package in ("vdjmatch", "vdjtools", "seqtree", "polars"):
    print(package, version(package))
# Open this notebook from the repository root or the examples directory.
examples = next((p for p in (Path.cwd()/"examples", Path.cwd())
                 if (p/"inputs/reference.tsv").is_file()), None)
if examples is None:
    raise RuntimeError("Use the downloaded complete examples bundle, not this notebook alone")
# Deterministic demonstration: no sampling or RNG seeds are used.
output = Path(tempfile.mkdtemp(prefix="vdjmatch-example-"))/"fresh"
output.mkdir()
print("Fresh output directory:", output)


In [ ]:
# Run the default vX workflow, optionally with a pinned official archive.
reference = os.environ.get("VDJMATCH_EXAMPLE_VDJDB")
reference_option = ["--vdjdb", str(Path(reference).expanduser().resolve())] if reference else []
prefix = output / "official"
subprocess.run(
    [sys.executable, "-m", "vdjmatch.cli", "annotate",
     str(examples / "inputs/cohort/sample_a.tsv"), "--locus", "TRB",
     "--threads", "4", "--output-prefix", str(prefix), *reference_option],
    check=True, timeout=120,
)

In [ ]:
# Verify one row per query and print the score and reference metadata.
import polars as pl

queries = pl.read_csv(prefix.with_suffix(".queries.tsv"), separator="	")
selection = pl.read_csv(prefix.with_suffix(".selection.tsv"), separator="	")
manifest = json.loads(prefix.with_suffix(".manifest.json").read_text())
assert queries["source_sequence_id"].to_list() == ["q1", "q2"]
assert selection.height == queries.height == 2
assert manifest["score_models"] == ["vx"]
assert manifest["exact_hits_excluded"] is False
assert manifest["reference_population"] == "curated-benchmark"
assert manifest["reference"]["input_sha256"]
print(json.dumps(manifest["reference"], indent=2))
print(queries.select("source_sequence_id", "status"))
print(selection.select("source_sequence_id", "selected_epitope", "selected_mhc_a",
                       "selected_mhc_b", "selected_mhc_class", "score", "selection_status"))

The output preserves the AIRR source ID for every query. A pMHC can appear when vX finds supported evidence and its selection policy resolves the label. Check the disposition before using the result. This example supplies no matched vX null bank. Use a matched bank with `--vx-null-dir` to report empirical E-values at a declared false-call operating point.